# Metro Interstate Traffic: End-to-End PySpark Pipeline
## Comprehensive Data Quality, Anomaly Diagnostics, Data Cleaning, Distributed EDA, and MLlib Regression Modeling

### Project Overview
This notebook implements an end-to-end Big Data pipeline for the **Metro Interstate Traffic Volume** dataset (westbound I-94 between Minneapolis and St. Paul, MN, 2012–2018) using **Apache PySpark**, **Spark SQL**, and **Spark MLlib**.

### Architecture & Pipeline Stages:
1. **Environment Setup & Session Initialization**: Configure local PySpark session with appropriate memory and shuffle partitions.
2. **Phase 1: Ingestion & Data Quality Assessment (`src/01_data_quality.py`)**: Audit schema, missing/NaN distributions, duplicate counts, and physical domain boundaries.
3. **Phase 1: Anomaly Diagnostics (`src/02_investigate_anomalies.py`)**: Root-cause analysis of 0 Kelvin temperatures, 9,831.3 mm rainfall glitch, highway closure events, and temporal span.
4. **Phase 2: Cleaning & Feature Engineering (`src/03_clean_data.py`)**: Exact deduplication, physical sensor sanitization, timestamp decomposition (`hour`, `day_of_week` with Sunday=1/Saturday=7, `month`, `is_weekend`), and Parquet export.
5. **Phase 3: Exploratory Data Analysis & Spark SQL (`src/04_traffic_analysis.py`)**: Diurnal rush-hour curves, weekday vs. weekend profiles, weather impacts, holiday reductions, and 300 DPI visualization exports.
6. **Phase 4: Machine Learning with Spark MLlib (`src/05_train_model.py`)**: Leak-free chronological 80/20 train/test split, MLlib Pipeline (`Imputer`, `StringIndexer`, `OneHotEncoder`, `VectorAssembler`), Baseline Linear Regression vs. Random Forest Regressor ($R^2 = 0.9120$), and model persistence.
7. **Phase 5: Interactive Streamlit Dashboard Deployment (`app.py`)**: Verification of live inference and web dashboard.


---
## 1. Environment Setup & SparkSession Initialization

We import the required PySpark, Spark MLlib, and visualization libraries. We initialize a `SparkSession` configured for local execution and set logging to `WARN`.


In [ ]:
import os
import sys
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, when, isnan, min, max, mean, avg, stddev,
    lit, hour, dayofweek, month, to_date, desc, asc, round as spark_round,
    unix_timestamp, expr
)
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import (
    Imputer, StringIndexer, OneHotEncoder, VectorAssembler
)
from pyspark.ml.regression import (
    LinearRegression, RandomForestRegressor
)
from pyspark.ml.evaluation import RegressionEvaluator

# Initialize SparkSession
spark = (
    SparkSession.builder
    .appName("TrafficPipelineEndToEnd")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("PySpark successfully initialized!")
print(f"Spark Version: {spark.version}")
print(f"Python: {sys.version.split()[0]}")


---
## 2. Data Ingestion & Schema Audit

We load `data/traffic_data.csv` with header extraction and automatic schema inference.


In [ ]:
# Dynamic path resolution to work from project root or execution directory
data_path = "data/traffic_data.csv" if os.path.exists("data/traffic_data.csv") else "../data/traffic_data.csv"
output_parquet = "output/traffic_cleaned.parquet" if os.path.exists("output") else "../output/traffic_cleaned.parquet"

print(f"Ingesting raw data from: {data_path}")
df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(data_path)
)
df_raw.cache()

raw_rows = df_raw.count()
raw_cols = len(df_raw.columns)
print(f"Dataset Dimensions: {raw_rows:,} rows | {raw_cols} columns")

print("\n--- INFERRED SCHEMA ---")
df_raw.printSchema()

print("\n--- SAMPLE FIRST 5 RECORDS ---")
df_raw.show(5, truncate=False)


---
## 3. Data Quality & Integrity Assessment (Reference: `src/01_data_quality.py`)

We perform rigorous quality audits:
- Null and NaN checks across floating-point and categorical columns.
- Exact duplicate detection.
- Continuous numerical statistics (`describe`).
- Snowfall distribution properties.
- Physical domain boundary checks.


In [ ]:
# 3.1 Missing and NaN value detection
missing_exprs = []
for field in df_raw.schema.fields:
    c = col(field.name)
    if field.dataType.typeName() in ("double", "float"):
        missing_exprs.append(count(when(c.isNull() | isnan(c), 1)).alias(field.name))
    else:
        missing_exprs.append(count(when(c.isNull(), 1)).alias(field.name))

print("=== MISSING / NaN VALUES PER COLUMN ===")
df_raw.select(*missing_exprs).show(truncate=False)

# 3.2 Exact Duplicate Rows
distinct_rows = df_raw.dropDuplicates().count()
duplicate_rows = raw_rows - distinct_rows
print(f"Total Rows:     {raw_rows:,}")
print(f"Distinct Rows:  {distinct_rows:,}")
print(f"Duplicate Rows: {duplicate_rows} ({duplicate_rows / raw_rows * 100:.2f}%)")

# 3.3 Numeric Feature Summary Statistics
print("\n=== NUMERIC SUMMARY STATISTICS ===")
df_raw.describe("temp", "rain_1h", "snow_1h", "clouds_all", "traffic_volume").show(truncate=False)

# 3.4 Snowfall Distribution Check
print("=== SNOWFALL DISTRIBUTION ===")
df_raw.select(
    count("*").alias("total_rows"),
    count(when(col("snow_1h") == 0, 1)).alias("zero_snow_rows"),
    count(when(col("snow_1h") > 0, 1)).alias("positive_snow_rows"),
    count(when(col("snow_1h") < 0, 1)).alias("negative_snow_rows"),
    max("snow_1h").alias("max_snowfall")
).show(truncate=False)

# 3.5 Domain Validity Checks
print("=== PHYSICAL DOMAIN VALIDITY CHECKS ===")
df_raw.select(
    count(when(col("traffic_volume") < 0, 1)).alias("negative_traffic"),
    count(when(col("rain_1h") < 0, 1)).alias("negative_rainfall"),
    count(when(col("snow_1h") < 0, 1)).alias("negative_snowfall"),
    count(when(col("clouds_all") < 0, 1)).alias("negative_clouds"),
    count(when(col("clouds_all") > 100, 1)).alias("clouds_over_100")
).show()


---
## 4. Anomaly Investigation & Diagnostics (Reference: `src/02_investigate_anomalies.py`)

We examine specific suspicious observations identified during data quality assessment:
1. **0.0 Kelvin Temperature**: Absolute zero sensor default state.
2. **Extreme Rainfall (9,831.3 mm)**: Sensor gauge glitch.
3. **Zero Traffic Volume Records**: Verified highway closures.
4. **Duplicate Record Grouping**: Inspecting exact copies.
5. **Temporal Span & Boundaries**: Accurate chronological span using `min()` and `max()`.


In [ ]:
# 4.1 Zero Temperature (0.0 Kelvin)
print("=== RECORDS WITH ZERO TEMPERATURE (0 Kelvin) ===")
zero_temp_df = df_raw.filter(col("temp") == 0)
print(f"Count of temp == 0.0 K: {zero_temp_df.count()}")
zero_temp_df.show(10, truncate=False)

# 4.2 Extreme Rainfall
print("\n=== TOP 10 HIGHEST RAINFALL RECORDS ===")
df_raw.orderBy(col("rain_1h").desc()).select(
    "date_time", "temp", "rain_1h", "snow_1h", "weather_main", "weather_description", "traffic_volume"
).show(10, truncate=False)

# 4.3 Zero Traffic Records
print("\n=== RECORDS WITH ZERO TRAFFIC VOLUME ===")
df_raw.filter(col("traffic_volume") == 0).select(
    "date_time", "temp", "rain_1h", "snow_1h", "weather_main", "traffic_volume"
).show(truncate=False)

# 4.4 Duplicated Complete Records
print("\n=== SAMPLE DUPLICATED COMPLETE RECORDS ===")
df_raw.groupBy(*df_raw.columns).agg(
    count("*").alias("copies")
).filter(col("copies") > 1).show(10, truncate=False)

# 4.5 Updated Temporal Span
print("\n=== TEMPORAL BOUNDARIES AND RECORD COUNTS ===")
df_raw.select(
    min("date_time").alias("earliest_timestamp"),
    max("date_time").alias("latest_timestamp"),
    count("*").alias("total_observations")
).show(truncate=False)


---
## 5. Data Cleaning & Feature Engineering (Reference: `src/03_clean_data.py`)

### Cleaning & Feature Engineering Rules:
1. **Deduplication**: Drop the 17 exact duplicate rows where all 9 columns match. Legitimate multi-weather events sharing the same timestamp are preserved.
2. **Sensor Sanitization**: Replace unphysical `temp == 0` and `rain_1h > 100` with `NULL`. Valid zeros in rainfall, snowfall, and traffic volume are strictly preserved.
3. **Temporal Features**: Extract `date`, `hour`, `day_of_week`, `month`, and `is_weekend`.
   * **Important Convention**: Spark SQL's `dayofweek()` standard designates **Sunday = 1** and **Saturday = 7**; weekdays are 2 through 6.
4. **Parquet Export**: Save the sanitized dataset to `output/traffic_cleaned.parquet`.


In [ ]:
# Step 1: Exact Deduplication
df_dedup = df_raw.dropDuplicates()
print(f"Rows after removing exact duplicates: {df_dedup.count():,} (Dropped: {df_raw.count() - df_dedup.count()})")

# Step 2: Sensor Value Sanitization (Replace with NULL for median imputer)
df_sanitized = df_dedup.withColumn(
    "temp",
    when(col("temp") == 0, lit(None).cast("double"))
    .otherwise(col("temp"))
).withColumn(
    "rain_1h",
    when(col("rain_1h") > 100, lit(None).cast("double"))
    .otherwise(col("rain_1h"))
)

# Step 3: Feature Engineering
# Note: Spark dayofweek() assigns Sunday = 1, Saturday = 7
df_cleaned = (
    df_sanitized
    .withColumn("date", to_date("date_time"))
    .withColumn("hour", hour("date_time"))
    .withColumn("day_of_week", dayofweek("date_time"))
    .withColumn("month", month("date_time"))
    .withColumn(
        "is_weekend",
        when(dayofweek("date_time").isin(1, 7), 1)
        .otherwise(0)
    )
)
df_cleaned.cache()

# Step 4: Post-Cleaning Audit
print("\n=== POST-CLEANING QUALITY AUDIT ===")
df_cleaned.select(
    count("*").alias("final_rows"),
    count(when(col("temp").isNull(), 1)).alias("sanitized_temp_nulls"),
    count(when(col("rain_1h").isNull(), 1)).alias("sanitized_rain_nulls"),
    count(when(col("traffic_volume") == 0, 1)).alias("preserved_zero_traffic"),
    count(when(col("snow_1h") == 0, 1)).alias("preserved_zero_snow")
).show(truncate=False)

# Step 5: Save Cleaned Parquet
os.makedirs(os.path.dirname(output_parquet) or ".", exist_ok=True)
df_cleaned.write.mode("overwrite").parquet(output_parquet)
print(f"Cleaned dataset successfully saved to: {output_parquet}")


---
## 6. Exploratory Data Analysis & Spark SQL Aggregations (Reference: `src/04_traffic_analysis.py`)

Using PySpark DataFrames and Spark SQL temp views, we compute aggregate traffic patterns:
1. Hourly volume profiles (peak rush hours).
2. Weekday vs. weekend commute contrast.
3. Monthly seasonality.
4. Weather impact analysis (avoiding causal fallacies).
5. Holiday vs. regular day traffic (handling string `'None'`).


In [ ]:
# Register SQL Temp View
df_cleaned.createOrReplaceTempView("traffic")

# 6.1 Hourly Traffic Volume
hourly_sql = """
SELECT 
    hour,
    ROUND(AVG(traffic_volume), 2) AS avg_traffic,
    ROUND(STDDEV(traffic_volume), 2) AS std_traffic,
    MIN(traffic_volume) AS min_traffic,
    MAX(traffic_volume) AS max_traffic,
    COUNT(*) AS record_count
FROM traffic
GROUP BY hour
ORDER BY hour
"""
hourly_df = spark.sql(hourly_sql)
print("=== HOURLY TRAFFIC PROFILE ===")
hourly_df.show(24, truncate=False)

# 6.2 Weekday vs Weekend Comparison
weekday_weekend_sql = """
SELECT 
    hour,
    is_weekend,
    CASE WHEN is_weekend = 1 THEN 'Weekend' ELSE 'Weekday' END AS day_type,
    ROUND(AVG(traffic_volume), 2) AS avg_traffic,
    COUNT(*) AS record_count
FROM traffic
GROUP BY hour, is_weekend
ORDER BY hour, is_weekend
"""
weekday_weekend_df = spark.sql(weekday_weekend_sql)
print("=== WEEKDAY vs. WEEKEND COMPARISON ===")
weekday_weekend_df.show(10, truncate=False)

# 6.3 Monthly Traffic Trends
monthly_sql = """
SELECT 
    month,
    ROUND(AVG(traffic_volume), 2) AS avg_traffic,
    ROUND(AVG(temp), 2) AS avg_temp_kelvin,
    ROUND(AVG(rain_1h), 3) AS avg_rain_mm,
    COUNT(*) AS record_count
FROM traffic
GROUP BY month
ORDER BY month
"""
monthly_df = spark.sql(monthly_sql)
print("=== MONTHLY SEASONAL TRENDS ===")
monthly_df.show(12, truncate=False)

# 6.4 Weather Category Impact
weather_sql = """
SELECT 
    weather_main,
    COUNT(*) AS record_count,
    ROUND(AVG(traffic_volume), 2) AS avg_traffic,
    ROUND(AVG(temp), 2) AS avg_temp_kelvin,
    ROUND(AVG(rain_1h), 3) AS avg_rain_mm
FROM traffic
GROUP BY weather_main
ORDER BY avg_traffic DESC
"""
weather_df = spark.sql(weather_sql)
print("=== TRAFFIC BY MAIN WEATHER CATEGORY ===")
weather_df.show(truncate=False)

# 6.5 Holiday vs Non-Holiday (Handling string 'None')
holiday_sql = """
SELECT 
    CASE WHEN holiday IS NULL OR holiday = 'None' THEN 'Regular Day' ELSE 'Holiday' END AS holiday_status,
    COUNT(*) AS record_count,
    ROUND(AVG(traffic_volume), 2) AS avg_traffic,
    MIN(traffic_volume) AS min_traffic,
    MAX(traffic_volume) AS max_traffic
FROM traffic
GROUP BY CASE WHEN holiday IS NULL OR holiday = 'None' THEN 'Regular Day' ELSE 'Holiday' END
"""
holiday_df = spark.sql(holiday_sql)
print("=== REGULAR DAYS vs. STATUTORY HOLIDAYS ===")
holiday_df.show(truncate=False)


---
## 7. Machine Learning Pipeline with Spark MLlib (Reference: `src/05_train_model.py`)

### Methodology:
1. **Chronological 80/20 Train/Test Split**: Strictly splits time-series observations at the 80th percentile timestamp (`2017-11-01 18:00:00`) to guarantee **zero temporal data leakage**.
2. **Spark MLlib Feature Pipeline**:
   - `Imputer`: Median imputation for `temp` and `rain_1h` nulls.
   - `StringIndexer` & `OneHotEncoder`: Encodes `weather_main` and `holiday`.
   - `VectorAssembler`: Assembles numeric and encoded categorical features.
3. **Model Evaluation**: Compares Baseline Linear Regression against Random Forest Regressor on out-of-sample test records using RMSE, MAE, and $R^2$.


In [ ]:
# 7.1 Chronological Split at 80% mark
cutoff_ts = df_cleaned.select(expr("percentile_approx(unix_timestamp(date_time), 0.80)")).collect()[0][0]

train_df = df_cleaned.filter(unix_timestamp("date_time") <= cutoff_ts).cache()
test_df = df_cleaned.filter(unix_timestamp("date_time") > cutoff_ts).cache()

print(f"Training observations: {train_df.count():,} (80%)")
print(f"Testing observations:  {test_df.count():,} (20%)")

# 7.2 Spark MLlib Pipeline Stages
imputer = Imputer(
    inputCols=["temp", "rain_1h"],
    outputCols=["temp_imputed", "rain_1h_imputed"],
    strategy="median"
)

weather_indexer = StringIndexer(inputCol="weather_main", outputCol="weather_main_idx", handleInvalid="keep")
holiday_indexer = StringIndexer(inputCol="holiday", outputCol="holiday_idx", handleInvalid="keep")

encoder = OneHotEncoder(
    inputCols=["weather_main_idx", "holiday_idx"],
    outputCols=["weather_main_vec", "holiday_vec"],
    handleInvalid="keep"
)

feature_cols = [
    "hour", "day_of_week", "month", "is_weekend",
    "temp_imputed", "rain_1h_imputed", "snow_1h", "clouds_all",
    "weather_main_vec", "holiday_vec"
]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# Models
lr = LinearRegression(featuresCol="features", labelCol="traffic_volume", maxIter=50, regParam=0.1)
rf = RandomForestRegressor(featuresCol="features", labelCol="traffic_volume", numTrees=60, maxDepth=12, seed=42)

lr_pipeline = Pipeline(stages=[imputer, weather_indexer, holiday_indexer, encoder, assembler, lr])
rf_pipeline = Pipeline(stages=[imputer, weather_indexer, holiday_indexer, encoder, assembler, rf])

# 7.3 Fit Models
print("\nFitting Baseline Linear Regression...")
lr_model = lr_pipeline.fit(train_df)

print("Fitting Random Forest Regressor (Trees=60, Depth=12)...")
rf_model = rf_pipeline.fit(train_df)
print("Models trained successfully!")

# 7.4 Evaluate on Out-of-Time Test Set
evaluator_rmse = RegressionEvaluator(labelCol="traffic_volume", predictionCol="prediction", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol="traffic_volume", predictionCol="prediction", metricName="mae")
evaluator_r2 = RegressionEvaluator(labelCol="traffic_volume", predictionCol="prediction", metricName="r2")

lr_preds = lr_model.transform(test_df)
rf_preds = rf_model.transform(test_df)

results_summary = pd.DataFrame([
    {"Model": "Baseline Linear Regression", "RMSE": round(evaluator_rmse.evaluate(lr_preds), 2), "MAE": round(evaluator_mae.evaluate(lr_preds), 2), "R2": round(evaluator_r2.evaluate(lr_preds), 4)},
    {"Model": "Random Forest Regressor", "RMSE": round(evaluator_rmse.evaluate(rf_preds), 2), "MAE": round(evaluator_mae.evaluate(rf_preds), 2), "R2": round(evaluator_r2.evaluate(rf_preds), 4)}
])

print("\n" + "=" * 65)
print("TEST SET EVALUATION METRICS COMPARISON")
print("=" * 65)
print(results_summary.to_string(index=False))

# Persist Pipeline
model_save_path = "output/models/rf_traffic_pipeline" if os.path.exists("output") else "../output/models/rf_traffic_pipeline"
rf_model.write().overwrite().save(model_save_path)
print(f"\nPersisted fitted Random Forest pipeline to: {model_save_path}")


---
## 8. Interactive Prediction Simulation & Streamlit Web App Deployment

We can simulate real-time inference using the persisted MLlib pipeline model. For the interactive web dashboard, run `streamlit run app.py` from your terminal.


In [ ]:
# Load persisted model and test single inference
loaded_model = PipelineModel.load(model_save_path)

sample_input = spark.createDataFrame([{
    "hour": 8,
    "day_of_week": 3,  # Tuesday (1=Sunday, 2=Monday, 3=Tuesday)
    "month": 10,
    "is_weekend": 0,
    "temp": 288.15,    # ~15 deg C
    "rain_1h": 0.0,
    "snow_1h": 0.0,
    "clouds_all": 40,
    "weather_main": "Clouds",
    "holiday": "None"
}])

prediction_res = loaded_model.transform(sample_input).select("prediction").collect()[0][0]
print(f"Simulated Commute Prediction: {int(round(prediction_res)):,} vehicles/hour")
print("Streamlit dashboard is available to launch: streamlit run app.py")


---
## 9. SparkSession Cleanup
Release Spark executor threads and JVM cluster memory.


In [ ]:
spark.stop()
print("SparkSession successfully closed.")
